# Customer Support RAG — Xumo Stream Box

Ye notebook `xumo_stream_box_guide.pdf` (setup, activation, troubleshooting, account & internet connectivity guide) par ek **Retrieval-Augmented Generation (RAG) based customer support bot** banata hai.

**Pipeline:**
`PDF Loader -> Text Splitter -> Open Source Embeddings -> FAISS Vector Store -> Retriever -> Prompt Template -> LLM -> Structured Output`

**Stack (LangChain v1, current as of Sep 2026):**
- `langchain-community` — `PyPDFLoader`, `FAISS`
- `langchain-text-splitters` — `RecursiveCharacterTextSplitter`
- `langchain-huggingface` — open source embeddings (`all-MiniLM-L6-v2`, no paid key)
- `langchain-groq` via `init_chat_model` — LLM for answer generation
- `langchain-core` — prompts, output parser, runnables, structured output

> Sirf LLM call ke liye ek free Groq API key chahiye (https://console.groq.com/keys). Baaki poora pipeline (PDF load, split, embed, FAISS, retriever) bina kisi key ke chalta hai.

## 0. Setup — Install, Imports, PDF Path

In [ ]:
!pip install -q -U langchain langchain-core langchain-text-splitters langchain-community langchain-huggingface langchain-groq faiss-cpu sentence-transformers pypdf

In [ ]:
import os
import warnings
warnings.filterwarnings("ignore")  # community deprecation notice — harmless, package still works

from getpass import getpass

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain.chat_models import init_chat_model

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from pydantic import BaseModel, Field
from typing import Literal

print("All imports successful ✅")

In [ ]:
# Path to the Xumo Stream Box PDF.
# Notebook ko PDF ke saath usi folder me rakho, ya yaha poora path daal do
# (jaise: "/Users/pradipwasre/Desktop/GenAI-V2/xumo_stream_box_guide.pdf")
PDF_PATH = "xumo_stream_box_guide.pdf"

assert os.path.exists(PDF_PATH), f"PDF not found at {PDF_PATH} — update PDF_PATH above."
print("PDF found:", PDF_PATH)

In [ ]:
# Groq API key (free tier: https://console.groq.com/keys)
if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your GROQ_API_KEY: ")

llm = init_chat_model("groq:llama-3.3-70b-versatile", temperature=0)
print(llm.invoke("Say hello in one short line").content)

## 1. Document Loader — Load the Xumo PDF

`PyPDFLoader` PDF ke har page ko ek alag `Document` object me load karta hai, page number metadata ke saath — isse baad me answer ke saath source page bhi bata sakte hain.

In [ ]:
loader = PyPDFLoader(PDF_PATH)
pdf_docs = loader.load()

print("Total pages loaded:", len(pdf_docs))
print("\nPage 1 preview:\n", pdf_docs[0].page_content[:300])
print("\nMetadata of page 1:", pdf_docs[0].metadata)

## 2. Text Splitter

Har page ko chhote overlapping chunks me todte hain taaki retrieval ke time precise, relevant context mile (poora page nahi, sirf relevant paragraph).

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120,
)

chunks = text_splitter.split_documents(pdf_docs)
print("Total chunks created:", len(chunks))
print("\nSample chunk:\n", chunks[5].page_content)
print("\nSource page:", chunks[5].metadata.get("page_label"))

## 3. Embeddings — Open Source Model

`sentence-transformers/all-MiniLM-L6-v2` — fast, lightweight, open source embedding model. Koi API key nahi chahiye, pehli baar chalane par model download hota hai.

In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

test_vector = embedding_model.embed_query("How do I fix my remote?")
print("Embedding dimension:", len(test_vector))

## 4. Vector Store — FAISS

Saare chunks ko embed karke FAISS index me store karte hain, aur disk par save bhi kar dete hain taaki dobara embed na karna pade.

In [ ]:
vectorstore = FAISS.from_documents(chunks, embedding_model)
vectorstore.save_local("xumo_faiss_index")

print("FAISS index built with", vectorstore.index.ntotal, "vectors")
print("Saved to ./xumo_faiss_index")

In [ ]:
# Reload check — confirms the saved index works independently of the build step
vectorstore = FAISS.load_local(
    "xumo_faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True,
)
print("Reloaded index, total vectors:", vectorstore.index.ntotal)

## 5. Retriever

Vector store ko ek `Runnable` retriever interface deते hain — top-4 most relevant chunks fetch karega har customer query ke liye.

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

test_results = retriever.invoke("activation code not working")
for i, doc in enumerate(test_results):
    print(f"--- Match {i+1} (page {doc.metadata.get('page_label')}) ---")
    print(doc.page_content[:200])
    print()

## 6. Prompt Template — Customer Support Persona

Support agent jaisa tone, sirf retrieved context use karke jawab dega, agar answer context me nahi hai toh clearly bolega aur customer ko live chat support ki taraf point karega.

In [ ]:
support_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a Xumo Stream Box customer support agent. "
     "Answer the customer's question using ONLY the context below. "
     "Be concise, friendly, and give clear step-by-step instructions when relevant. "
     "If the answer is not in the context, say you don't have that information and "
     "suggest contacting Xumo support via Start Chat — do not make anything up.\n\n"
     "Context:\n{context}"),
    ("human", "{question}"),
])

def format_docs(docs):
    return "\n\n".join(
        f"[Page {d.metadata.get('page_label')}] {d.page_content}" for d in docs
    )

## 7. Structured Output

Har support response ko sirf plain text nahi, balki ek structured ticket-style object me bhi chahiye — jisme category, escalation flag, aur confidence ho. Isse ye output kisi bhi support dashboard/CRM me directly plug ho sakta hai.

In [ ]:
class SupportResponse(BaseModel):
    """Structured customer support response for a Xumo Stream Box query."""
    answer: str = Field(description="Clear, step-by-step answer to the customer's question")
    category: Literal[
        "activation", "wifi_connectivity", "remote_issue", "account_login",
        "hardware_issue", "software_update", "other"
    ] = Field(description="Best matching support category for this query")
    needs_human_escalation: bool = Field(
        description="True if the answer was not found in context and a human agent should step in"
    )

structured_llm = llm.with_structured_output(SupportResponse)

structured_prompt = support_prompt  # same context+question format, different terminal step

def build_structured_chain():
    return (
        RunnableParallel(
            context=retriever | RunnableLambda(format_docs),
            question=RunnablePassthrough(),
        )
        | structured_prompt
        | structured_llm
    )

structured_chain = build_structured_chain()

## 8. Output Parser + LLM — Plain-Text RAG Chain

Ek simple plain-text version bhi banate hain (jaise ek chatbot widget me dikhana ho) — `StrOutputParser` se LLM ka raw `AIMessage` clean string me convert ho jata hai.

In [ ]:
rag_chain = (
    RunnableParallel(
        context=retriever | RunnableLambda(format_docs),
        question=RunnablePassthrough(),
    )
    | support_prompt
    | llm
    | StrOutputParser()
)

## 9. Test the Xumo Support Bot

Kuch real customer-style questions PDF ke content se test karte hain.

In [ ]:
test_questions = [
    "My activation code is not working, what should I do?",
    "The screen is black, how do I fix it?",
    "How do I connect my device to a new WiFi network?",
    "Do I need a Xumo account to use the Stream Box?",
    "What is the capital of France?",  # out-of-scope question to test guardrails
]

for q in test_questions:
    print("Q:", q)
    print("A:", rag_chain.invoke(q))
    print("-" * 80)

In [ ]:
# Same questions through the structured output version
for q in test_questions[:3]:
    result = structured_chain.invoke(q)
    print("Q:", q)
    print(result)
    print("-" * 80)

## 10. Interactive Support Chat (Optional)

Ye cell run karke aap khud live query type kar sakte ho. Exit karne ke liye `exit` ya `quit` type karo.

In [ ]:
def ask_xumo_support(query: str) -> str:
    """Simple helper function to query the Xumo support RAG chain."""
    return rag_chain.invoke(query)

# Example single call:
# print(ask_xumo_support("My remote is not responding, how do I fix it?"))

while True:
    user_query = input("Ask Xumo Support (or type 'exit'): ")
    if user_query.strip().lower() in ("exit", "quit"):
        print("Chat ended.")
        break
    print("Bot:", ask_xumo_support(user_query))
    print()

### Recap

| Step | Component | Notes |
|---|---|---|
| Load | `PyPDFLoader` | 1 Document per PDF page |
| Split | `RecursiveCharacterTextSplitter` | 800 char chunks, 120 overlap |
| Embed | `HuggingFaceEmbeddings` | open source, `all-MiniLM-L6-v2` |
| Store | `FAISS` | saved locally to `./xumo_faiss_index` |
| Retrieve | `vectorstore.as_retriever()` | top-4 similarity search |
| Generate | `init_chat_model` (Groq) | grounded, context-only answers |
| Structured Output | `with_structured_output` | category + escalation flag for CRM use |
| Output Parser | `StrOutputParser` | plain text for chatbot widgets |

Real customer support product me isi pattern ko extend kar sakte ho: multiple PDFs load karo, `FAISS.add_documents()` se index update karo, aur `needs_human_escalation=True` wale tickets ko live agent ko route kar do.